# 04 — SparkSQL

**Airline Operations Intelligence Platform** · Notebook 4 of 10 · *runs locally*

## Purpose
Demonstrate SparkSQL as required by **Module 4** of the project plan:

- Register cleaned DataFrames as temporary SQL views
- Run SQL aggregation queries over 5.8M records
- Show the equivalent DataFrame API code **side by side**
- Prove that SparkSQL and the DataFrame API produce **identical results**

Syllabus coverage (Unit 4): SparkSQL, Spark ecosystem, Catalyst optimisation,
lazy evaluation, DAG, caching.

## Why this matters for the project
Every aggregation in notebook `05` could be written either way. This notebook establishes
that the choice is one of readability, not correctness or speed — both compile to the same
optimised plan, as verified in notebook `03`.

In [ ]:
import sys, time
sys.path.insert(0, "../src")

from config import build_spark, PATHS
from pyspark.sql import functions as F

spark = build_spark("04-sparksql")

flights = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
flights.cache()
N = flights.count()
print(f"Curated flights: {N:,}")

---
## 1. Registering temporary views

A **temporary view** exposes a DataFrame to the SQL engine. It is scoped to this
`SparkSession` and holds no data of its own — it is a name bound to a logical plan, so
querying it re-reads the same cached DataFrame.

In [ ]:
flights.createOrReplaceTempView("flights")

# Dimension views, so joins can be demonstrated in SQL.
(flights.select("airline_code", "airline_name").distinct()
        .createOrReplaceTempView("airlines"))

(flights.select(F.col("origin").alias("code"), F.col("origin_name").alias("name"),
                F.col("origin_city").alias("city"), F.col("origin_state").alias("state"),
                F.col("origin_lat").alias("lat"), F.col("origin_lon").alias("lon"))
        .distinct().createOrReplaceTempView("airports"))

spark.sql("SHOW TABLES").show()

In [ ]:
spark.sql("DESCRIBE flights").show(60, truncate=False)

---
## 2. The plan's reference query

Module 4 of the project plan specifies this query. Run it first, exactly as written.

In [ ]:
spark.sql("""
    SELECT airline_code,
           ROUND(AVG(dep_delay), 2) AS avg_departure_delay
    FROM flights
    WHERE status = 'completed'
    GROUP BY airline_code
    ORDER BY avg_departure_delay DESC
""").show()

---
## 3. SQL and DataFrame API, side by side

The core requirement of Module 4: the same question answered both ways, with the results
asserted equal rather than eyeballed.

In [ ]:
# ---------- SparkSQL ----------
sql_result = spark.sql("""
    SELECT airline_code,
           COUNT(*)                                   AS total_flights,
           ROUND(AVG(dep_delay), 2)                   AS avg_dep_delay,
           ROUND(AVG(arr_delay), 2)                   AS avg_arr_delay,
           ROUND(100.0 * AVG(is_delayed), 2)          AS delay_rate_pct
    FROM flights
    WHERE status = 'completed'
    GROUP BY airline_code
    ORDER BY delay_rate_pct DESC
""")

# ---------- DataFrame API ----------
df_result = (flights
    .filter(F.col("status") == "completed")
    .groupBy("airline_code")
    .agg(F.count("*").alias("total_flights"),
         F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
         F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
         F.round(100.0 * F.avg("is_delayed"), 2).alias("delay_rate_pct"))
    .orderBy(F.desc("delay_rate_pct")))

sql_result.show()

In [ ]:
# Prove equivalence rather than assuming it.
a = [r.asDict() for r in sql_result.collect()]
b = [r.asDict() for r in df_result.collect()]

assert a == b, "SparkSQL and DataFrame API disagree"
print(f"IDENTICAL - {len(a)} rows match exactly across both APIs.")
print("\nThe choice between them is stylistic. Catalyst compiles both to the same plan.")

---
## 4. Joins in SQL

Notebook 02 already denormalised the dataset, so these joins are demonstrations of the
SQL surface rather than pipeline necessities — the dashboard reads pre-joined documents.

In [ ]:
spark.sql("""
    SELECT ap.name        AS airport,
           ap.city,
           ap.state,
           COUNT(*)                          AS departures,
           ROUND(AVG(f.dep_delay), 2)        AS avg_dep_delay,
           ROUND(100.0 * AVG(f.is_delayed), 2) AS delay_rate_pct
    FROM flights f
    JOIN airports ap ON f.origin = ap.code
    WHERE f.status = 'completed'
    GROUP BY ap.name, ap.city, ap.state
    HAVING COUNT(*) >= 10000
    ORDER BY delay_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

### `HAVING` as a bias control

The `HAVING COUNT(*) >= 10000` clause is not cosmetic. It implements the **small-sample
bias** mitigation the project plan commits to (§18 of the proposal): an airport with 40
flights can top a delay ranking on noise alone. Filtering by volume before ranking is how
the SQL layer enforces fair comparison.

Demonstrated below — the same query without the threshold.

In [ ]:
unfiltered = spark.sql("""
    SELECT origin, COUNT(*) AS flights,
           ROUND(100.0 * AVG(is_delayed), 2) AS delay_rate_pct
    FROM flights WHERE status = 'completed'
    GROUP BY origin ORDER BY delay_rate_pct DESC LIMIT 5
""")

filtered = spark.sql("""
    SELECT origin, COUNT(*) AS flights,
           ROUND(100.0 * AVG(is_delayed), 2) AS delay_rate_pct
    FROM flights WHERE status = 'completed'
    GROUP BY origin HAVING COUNT(*) >= 10000
    ORDER BY delay_rate_pct DESC LIMIT 5
""")

print("WITHOUT a minimum-sample threshold -- rankings driven by tiny airports:")
unfiltered.show()
print("WITH  HAVING COUNT(*) >= 10000 -- defensible ranking:")
filtered.show()

---
## 5. Window functions

Ranking without collapsing rows. These power the dashboard's "top N per group" views.

In [ ]:
spark.sql("""
    WITH route_stats AS (
        SELECT route, origin, destination,
               COUNT(*)                            AS flights,
               ROUND(AVG(arr_delay), 2)            AS avg_delay,
               ROUND(100.0 * AVG(is_delayed), 2)   AS delay_rate_pct
        FROM flights
        WHERE status = 'completed'
        GROUP BY route, origin, destination
        HAVING COUNT(*) >= 1000
    )
    SELECT route, flights, avg_delay, delay_rate_pct,
           RANK() OVER (ORDER BY delay_rate_pct DESC) AS worst_rank,
           RANK() OVER (ORDER BY delay_rate_pct ASC)  AS best_rank
    FROM route_stats
    ORDER BY worst_rank
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Per-group ranking: the worst origin airport within each state.
spark.sql("""
    SELECT state, origin, flights, delay_rate_pct FROM (
        SELECT ap.state, f.origin,
               COUNT(*) AS flights,
               ROUND(100.0 * AVG(f.is_delayed), 2) AS delay_rate_pct,
               ROW_NUMBER() OVER (PARTITION BY ap.state
                                  ORDER BY AVG(f.is_delayed) DESC) AS rn
        FROM flights f JOIN airports ap ON f.origin = ap.code
        WHERE f.status = 'completed'
        GROUP BY ap.state, f.origin
        HAVING COUNT(*) >= 5000
    )
    WHERE rn = 1
    ORDER BY delay_rate_pct DESC
    LIMIT 10
""").show()

---
## 6. Time-based analysis

Answers two of the questions the project plan sets out: *at what times and on which days
do delays increase?*

In [ ]:
spark.sql("""
    SELECT sched_dep_hour AS hour,
           COUNT(*) AS flights,
           ROUND(100.0 * AVG(is_delayed), 2) AS delay_rate_pct,
           ROUND(AVG(dep_delay), 2)          AS avg_dep_delay
    FROM flights
    WHERE status = 'completed'
    GROUP BY sched_dep_hour
    ORDER BY hour
""").show(24)

In [ ]:
spark.sql("""
    SELECT season,
           CASE day_of_week
                WHEN 1 THEN '1 Mon' WHEN 2 THEN '2 Tue' WHEN 3 THEN '3 Wed'
                WHEN 4 THEN '4 Thu' WHEN 5 THEN '5 Fri' WHEN 6 THEN '6 Sat'
                ELSE '7 Sun' END AS day,
           ROUND(100.0 * AVG(is_delayed), 2) AS delay_rate_pct
    FROM flights
    WHERE status = 'completed'
    GROUP BY season, day_of_week
    ORDER BY season, day
""").show(30)

---
## 7. Delay causes

The five cause columns are populated only for flights arriving 15+ minutes late
(notebook 01, rule 3). The `WHERE` clause below respects that — averaging across all
flights would divide by the wrong denominator.

In [ ]:
spark.sql("""
    SELECT COUNT(*)                                AS late_flights,
           ROUND(AVG(delay_carrier), 2)            AS carrier,
           ROUND(AVG(delay_weather), 2)            AS weather,
           ROUND(AVG(delay_nas), 2)                AS nas,
           ROUND(AVG(delay_security), 2)           AS security,
           ROUND(AVG(delay_late_aircraft), 2)      AS late_aircraft
    FROM flights
    WHERE status = 'completed' AND arr_delay >= 15
""").show()

print("Minutes attributable to each cause, averaged over flights 15+ min late.")

In [ ]:
# Cancellations by decoded reason.
spark.sql("""
    SELECT cancellation_reason,
           COUNT(*) AS cancellations,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_cancellations
    FROM flights
    WHERE status = 'cancelled'
    GROUP BY cancellation_reason
    ORDER BY cancellations DESC
""").show()

---
## 8. Catalyst optimisation, visible in SQL

Two features worth showing explicitly, because they are the reason SparkSQL over Parquet
is fast: **predicate pushdown** and **column pruning**.

In [ ]:
q = spark.sql("""
    SELECT airline_code, AVG(arr_delay)
    FROM flights
    WHERE month = 7 AND status = 'completed'
    GROUP BY airline_code
""")
q.explain(mode="formatted")

In [ ]:
import textwrap

# IMPORTANT: `flights` is cached in memory from section 1. Spark's cache manager
# substitutes a cached relation for any scan of the same files, so the query would
# read from memory and NO file-level pruning would appear in the plan. Clear the
# cache first, otherwise this demonstration silently measures the wrong thing.
spark.catalog.clearCache()

disk = spark.read.parquet(str(PATHS["curated"] / "flights.parquet"))
disk.createOrReplaceTempView("flights_disk")

q = spark.sql("""
    SELECT airline_code, AVG(arr_delay) FROM flights_disk
    WHERE month = 7 AND status = 'completed' GROUP BY airline_code
""")
q.collect()          # force the final adaptive plan
plan = q._jdf.queryExecution().executedPlan().toString()

print("Physical scan node Catalyst produced:\n")
for line in plan.split("\n"):
    if "FileScan" in line:
        body = line.strip()
        for field in ["PartitionFilters", "PushedFilters", "ReadSchema"]:
            body = body.replace(field, "\n" + field)
        print(textwrap.fill(body.split("\n")[0], width=98,
                            initial_indent="  ", subsequent_indent="      "))
        for part in body.split("\n")[1:]:
            print(textwrap.fill(part, width=98,
                                initial_indent="    ", subsequent_indent="        "))
        break

In [ ]:
print("Three optimisations, all derived by Catalyst from the SQL:\n")
print("  PartitionFilters : month = 7 prunes 11 of 12 partition directories.")
print("                     Those files are never opened at all.")
print("  PushedFilters    : status = 'completed' is evaluated inside the Parquet")
print("                     reader, so non-matching row groups never reach Spark.")
print(f"  ReadSchema       : only the 3 columns the query needs, of {len(disk.columns)}.")
print("                     Parquet is columnar, so the other 46 are never read.\n")
print("Re-cache for the sections that follow.")
flights.cache().count()

Three optimisations are visible above:

- **PartitionFilters** — `month = 7` prunes 11 of 12 partition directories at the file
  level. Those files are never opened.
- **PushedFilters** — `status = 'completed'` is evaluated inside the Parquet reader, so
  non-matching row groups are skipped before any row reaches Spark.
- **ReadSchema** — only `airline_code`, `arr_delay` and `status` are read. The other
  46 columns are never touched, because Parquet is columnar.

None of this was written by hand. Catalyst derived it from the SQL.

---
## 9. Caching a SQL result

Views are logical, not materialised. When a result is queried repeatedly — as the
dashboard will — caching it avoids recomputation.

In [ ]:
airline_summary = spark.sql("""
    SELECT airline_code, airline_name,
           COUNT(*) AS flights,
           ROUND(AVG(arr_delay), 2) AS avg_arr_delay,
           ROUND(100.0 * AVG(is_delayed), 2) AS delay_rate_pct
    FROM flights WHERE status = 'completed'
    GROUP BY airline_code, airline_name
""")

t0 = time.time(); airline_summary.count(); t_cold = time.time() - t0

airline_summary.cache(); airline_summary.count()          # materialise
t0 = time.time(); airline_summary.count(); t_warm = time.time() - t0

print(f"Uncached recompute : {t_cold:.2f}s")
print(f"Cached             : {t_warm:.3f}s")
print(f"Speedup            : {t_cold/max(t_warm,1e-6):.0f}x")
print("\nThis is the principle behind the serving layer: compute once in Spark,")
print("store the small result, and let the dashboard read it repeatedly.")

---
## 10. Summary

| Requirement (Module 4) | Demonstrated in |
|---|---|
| Register DataFrames as temporary SQL views | §1 |
| SQL aggregation with `GROUP BY` / `ORDER BY` | §2, §3 |
| Equivalent DataFrame API code side by side | §3 |
| Results proven identical | §3 — asserted, not eyeballed |
| Joins | §4 |
| Window functions and CTEs | §5 |
| Catalyst optimisation | §8 |

**Conclusion for the report:** SparkSQL and the DataFrame API are two front-ends to one
engine. SQL is more readable for multi-level aggregation and ranking; the DataFrame API
composes better inside Python. Notebook `05` uses the DataFrame API for pipeline code and
SQL where a query reads more clearly as SQL.

In [ ]:
flights.unpersist()
spark.stop()
print("Notebook 04 complete.")